# Week 6 — Preprocessing Report

**Project:** Mining EV Registration Patterns in Washington State  
**Data:** 50,000-row random sample (seed 42)

The sample was created from 299,705 records in `00_make_sample.ipynb` and was also used in Weeks 4 and 5. This notebook prepares data for clustering, association rule mining, and anomaly detection.

In [10]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/ev_sample_50k.csv", sep="\t")
df_clean = df.copy()

print("Sample shape:", df.shape)

Sample shape: (50000, 16)


## 1. Transformations

Postal codes are stored as five-character strings. Zero and missing electric ranges remain in the original column, but are excluded from range averages.

In [11]:
df_clean["Postal Code"] = (
    df_clean["Postal Code"].astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

df_clean["Electric Range Usable"] = df_clean["Electric Range"].where(
    df_clean["Electric Range"] > 0
)

print("Missing ZIP codes:", df_clean["Postal Code"].isna().sum())
print("Positive ranges:", df_clean["Electric Range Usable"].notna().sum())
print("Zero or missing ranges:", df_clean["Electric Range Usable"].isna().sum())

Missing ZIP codes: 2
Positive ranges: 17243
Zero or missing ranges: 32757


## 2. Feature Creation

For ZIP-code analysis, use Washington records with valid ZIP codes and create BEV/PHEV indicators. The full 50,000-row working dataset is retained.

In [12]:
df_wa = df_clean.loc[
    (df_clean["State"] == "WA") &
    df_clean["Postal Code"].str.fullmatch(r"\d{5}").fillna(False)
].copy()

df_wa["Is BEV"] = (
    df_wa["Electric Vehicle Type"] == "Battery Electric Vehicle (BEV)"
).astype(int)
df_wa["Is PHEV"] = (
    df_wa["Electric Vehicle Type"] == "Plug-in Hybrid Electric Vehicle (PHEV)"
).astype(int)

print("Washington records:", len(df_wa))
print("Unique ZIP codes:", df_wa["Postal Code"].nunique())
print("BEVs / PHEVs:", df_wa["Is BEV"].sum(), "/", df_wa["Is PHEV"].sum())

Washington records: 49873
Unique ZIP codes: 509
BEVs / PHEVs: 40168 / 9705


## 3. ZIP-Code Aggregation

Each ZIP code is represented by its EV count, vehicle-type counts, average model year, and average positive electric range.

In [13]:
zip_summary = df_wa.groupby("Postal Code").agg(
    EV_Count=("DOL Vehicle ID", "count"),
    BEV_Count=("Is BEV", "sum"),
    PHEV_Count=("Is PHEV", "sum"),
    Avg_Model_Year=("Model Year", "mean"),
    Avg_Electric_Range=("Electric Range Usable", "mean"),
    Valid_Range_Count=("Electric Range Usable", "count")
).reset_index()

zip_summary["BEV_Percentage"] = (
    zip_summary["BEV_Count"] / zip_summary["EV_Count"] * 100
).round(2)

print("ZIP codes:", len(zip_summary))
print("Aggregated vehicles:", zip_summary["EV_Count"].sum())
display(zip_summary.head())

ZIP codes: 509
Aggregated vehicles: 49873


,Postal Code,EV_Count,BEV_Count,PHEV_Count,Avg_Model_Year,Avg_Electric_Range,Valid_Range_Count,BEV_Percentage
0,98001,225,196,29,2022.648889,118.105263,57,87.11
1,98002,92,64,28,2021.869565,87.152174,46,69.57
2,98003,193,157,36,2022.336788,99.904762,63,81.35
3,98004,662,583,79,2022.758308,135.401130,177,88.07
4,98005,355,307,48,2023.177465,118.236559,93,86.48


## 4. Scaling

We standardize four numeric ZIP-level features. ZIP codes without a positive range have their average range filled with the observed ZIP-level median; a flag records which values were missing.

In [14]:
features = [
    "EV_Count", "BEV_Percentage", "Avg_Model_Year", "Avg_Electric_Range"
]

zip_features = zip_summary[features].copy()
range_missing = zip_features["Avg_Electric_Range"].isna().astype(int)
median_range = zip_features["Avg_Electric_Range"].median()
zip_features["Avg_Electric_Range"] = (
    zip_features["Avg_Electric_Range"].fillna(median_range)
)

scaler = StandardScaler()
zip_scaled = pd.DataFrame(
    scaler.fit_transform(zip_features),
    columns=features,
    index=zip_summary.index
)
zip_scaled.insert(0, "Postal Code", zip_summary["Postal Code"])
zip_scaled["Range_Missing"] = range_missing

print("ZIP codes with missing mean range:", range_missing.sum())
print("Median range used (miles):", round(median_range, 2))
print("Scaled dataset shape:", zip_scaled.shape)
display(zip_scaled.head())

ZIP codes with missing mean range: 51
Median range used (miles): 102.0
Scaled dataset shape: (509, 6)


,Postal Code,EV_Count,BEV_Percentage,Avg_Model_Year,Avg_Electric_Range,Range_Missing
0,98001,0.842328,0.469886,0.419143,0.220847,0
1,98002,-0.039672,-0.388303,-0.017213,-0.402559,0
2,98003,0.630117,0.188063,0.244393,-0.145718,0
3,98004,3.740328,0.516856,0.480409,0.569191,0
4,98005,1.704433,0.439061,0.715102,0.223491,0


## 5. Verification

Check that no vehicles were lost during ZIP aggregation and that the four scaled features contain no missing values.

In [15]:
assert df.shape == (50000, 16)
assert zip_summary["EV_Count"].sum() == len(df_wa)
assert len(zip_scaled) == len(zip_summary)
assert not zip_scaled.isna().any().any()

print("Sample rows:", len(df))
print("Washington rows in aggregation:", len(df_wa))
print("ZIP codes in scaled dataset:", len(zip_scaled))
print("Missing range flags:", zip_scaled["Range_Missing"].sum())
print("Scaled feature means:", zip_scaled[features].mean().round(4).to_dict())
print("Scaled feature std (ddof=0):", zip_scaled[features].std(ddof=0).round(4).to_dict())
print("Verification passed")

Sample rows: 50000
Washington rows in aggregation: 49873
ZIP codes in scaled dataset: 509
Missing range flags: 51
Scaled feature means: {'EV_Count': -0.0, 'BEV_Percentage': 0.0, 'Avg_Model_Year': 0.0, 'Avg_Electric_Range': -0.0}
Scaled feature std (ddof=0): {'EV_Count': 1.0, 'BEV_Percentage': 1.0, 'Avg_Model_Year': 1.0, 'Avg_Electric_Range': 1.0}
Verification passed


## 6. Vehicle-Level Feature Preparation

Separate vehicle-level features are prepared for association rule mining and anomaly detection.

In [16]:
# Features for association rule mining

association_data = df_clean[
    [
        "Electric Vehicle Type",
        "Make",
        "Model Year",
        "Clean Alternative Fuel Vehicle (CAFV) Eligibility"
    ]
].copy()

# Group model years into five-year intervals
year_start = (association_data["Model Year"] // 5) * 5

association_data["Year Group"] = (
    year_start.astype(str) + "-" +
    (year_start + 4).astype(str)
)

association_data = association_data.drop(columns=["Model Year"])

# Encode categorical features
association_encoded = pd.get_dummies(
    association_data,
    dtype=int
)

# Features for anomaly detection
anomaly_data = df_clean.loc[
    df_clean["Electric Range Usable"].notna(),
    [
        "DOL Vehicle ID",
        "Electric Vehicle Type",
        "Model Year",
        "Electric Range Usable"
    ]
].copy()

print("Association records:", len(association_data))
print("Encoded association features:", association_encoded.shape[1])
print("Anomaly records with available ranges:", len(anomaly_data))

display(association_data.head())
display(anomaly_data.head())

Association records: 50000
Encoded association features: 59
Anomaly records with available ranges: 17243


,Electric Vehicle Type,Make,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Year Group
0,Battery Electric Vehicle (BEV),LEXUS,Eligibility unknown as battery range has not b...,2020-2024
1,Battery Electric Vehicle (BEV),HYUNDAI,Eligibility unknown as battery range has not b...,2025-2029
2,Battery Electric Vehicle (BEV),VOLKSWAGEN,Eligibility unknown as battery range has not b...,2025-2029
3,Battery Electric Vehicle (BEV),RIVIAN,Eligibility unknown as battery range has not b...,2020-2024
4,Battery Electric Vehicle (BEV),TESLA,Eligibility unknown as battery range has not b...,2020-2024


,DOL Vehicle ID,Electric Vehicle Type,Model Year,Electric Range Usable
11,276243026,Battery Electric Vehicle (BEV),2020,291.0
12,476992141,Battery Electric Vehicle (BEV),2019,220.0
14,103214044,Battery Electric Vehicle (BEV),2020,259.0
18,294698974,Plug-in Hybrid Electric Vehicle (PHEV),2021,20.0
22,269409304,Battery Electric Vehicle (BEV),2019,220.0


## 7. Preprocessing Summary

The 50,000-record sample was used for preprocessing. Postal codes were converted to five-digit strings, and unavailable electric ranges were identified without removing the original records.

For clustering, 49,873 Washington records were aggregated into 509 ZIP codes. Missing average electric ranges were filled using the median, and four numerical features were standardized.

For association rule mining, 50,000 vehicle records were prepared using categorical features, resulting in 59 encoded attributes. For anomaly detection, 17,243 records with positive electric ranges were retained for further analysis.

The preprocessing results provide separate datasets for the three proposed mining methods. Records with zero or missing electric range remain in the working dataset but are excluded from range-based anomaly analysis.